In [9]:
HUB_REPO = "suellenlima/bertimbau-sentimento-reviews-ptbr"
PASTA = "onnx_export"

In [10]:
%pip install -q "transformers>=5.0" onnx onnxruntime

In [11]:
import os

import numpy as np
import onnxruntime as ort
import torch
from huggingface_hub import HfApi, login
from onnxruntime.quantization import QuantType, quantize_dynamic
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [12]:
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except ImportError:
    login()

In [13]:
os.makedirs(f"{PASTA}/onnx", exist_ok=True)

modelo = AutoModelForSequenceClassification.from_pretrained(HUB_REPO).eval()
tokenizer = AutoTokenizer.from_pretrained(HUB_REPO)


class SoLogits(torch.nn.Module):
    """Envolve o modelo para que a saída seja só os logits, como o navegador espera."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask, token_type_ids):
        return self.m(input_ids=input_ids, attention_mask=attention_mask,
                      token_type_ids=token_type_ids).logits


entradas = ["input_ids", "attention_mask", "token_type_ids"]
exemplo = tokenizer(["A entrega chegou atrasada.", "Adorei"], padding=True, return_tensors="pt")
eixos = {nome: {0: "batch", 1: "sequencia"} for nome in entradas}
eixos["logits"] = {0: "batch"}

torch.onnx.export(
    SoLogits(modelo).eval(),
    tuple(exemplo[n] for n in entradas),
    f"{PASTA}/onnx/model.onnx",
    input_names=entradas,
    output_names=["logits"],
    dynamic_axes=eixos,
    opset_version=17,
    dynamo=False,
)
modelo.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_425/86305385.py:24: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [14]:
%pip install -q onnxconverter-common

In [15]:
import onnx
from onnxconverter_common import float16

frases_teste = [
    "Entrega rápida, comida quentinha!", "Pedido veio errado e frio.", "Parabéns pela demora.",
    "Produto excelente, recomendo muito.", "Não recebi o produto até hoje.", "Chegou antes do prazo, adorei.",
    "Veio quebrado e ninguém responde.", "Produto bom, mas a entrega atrasou.", "Muito satisfeita com a compra!",
    "Péssimo atendimento, nunca mais compro.", "Tudo certo, chegou direitinho.", "Demorou mas entregou!",
]
tok_teste = tokenizer(frases_teste, padding=True, return_tensors="np")
with torch.no_grad():
    ref = torch.softmax(modelo(**tokenizer(frases_teste, padding=True, return_tensors="pt")).logits, dim=-1).numpy()


def conferir(caminho):
    """Compara a versão ONNX com o modelo original nas frases de teste."""
    sessao = ort.InferenceSession(caminho)
    logits = sessao.run(["logits"], {n: tok_teste[n].astype(np.int64) for n in entradas})[0]
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    concordam = int((probs.argmax(-1) == ref.argmax(-1)).sum())
    return concordam, float(np.abs(probs - ref).max())


original = f"{PASTA}/onnx/model.onnx"
candidatos = []

# 1) 8 bits, só pesos fixos, por canal, faixa reduzida (~110 MB)
quantize_dynamic(original, f"{PASTA}/onnx/model_quantized.onnx", weight_type=QuantType.QInt8,
                 per_channel=True, reduce_range=True, extra_options={"MatMulConstBOnly": True})
candidatos.append(("q8", "model_quantized.onnx"))

# 2) 8 bits sem sinal, só pesos fixos (~110 MB)
quantize_dynamic(original, f"{PASTA}/onnx/model_uint8.onnx", weight_type=QuantType.QUInt8,
                 extra_options={"MatMulConstBOnly": True})
candidatos.append(("uint8", "model_uint8.onnx"))

# 3) 16 bits (~220 MB): metade do tamanho, quase sem perda
onnx.save(float16.convert_float_to_float16(onnx.load(original), keep_io_types=True),
          f"{PASTA}/onnx/model_fp16.onnx")
candidatos.append(("fp16", "model_fp16.onnx"))

# 4) original em 32 bits (~436 MB): sem perda nenhuma
candidatos.append(("fp32", "model.onnx"))

escolhido = None
print(f"{'versão':7s} {'arquivo':24s} {'tamanho':>8s} {'concordam':>10s} {'maior diferença':>16s}")
for dtype, arquivo in candidatos:
    caminho = f"{PASTA}/onnx/{arquivo}"
    concordam, dif = conferir(caminho)
    mb = os.path.getsize(caminho) / 1e6
    print(f"{dtype:7s} {arquivo:24s} {mb:6.0f} MB {concordam:>5d}/{len(frases_teste)} {dif:16.1%}")
    if escolhido is None and concordam == len(frases_teste) and dif < 0.05:
        escolhido = (dtype, arquivo)

DTYPE, ARQUIVO = escolhido
print(f"\nEscolhido: {DTYPE} ({ARQUIVO})")

versão  arquivo                   tamanho  concordam  maior diferença
q8      model_quantized.onnx        110 MB    12/12             3.0%
uint8   model_uint8.onnx            110 MB    12/12             5.4%
fp16    model_fp16.onnx             218 MB    12/12             0.0%
fp32    model.onnx                  436 MB    12/12             0.0%

Escolhido: q8 (model_quantized.onnx)


In [16]:
HfApi().upload_file(
    path_or_fileobj=f"{PASTA}/onnx/{ARQUIVO}",
    path_in_repo=f"onnx/{ARQUIVO}",
    repo_id=HUB_REPO,
    commit_message=f"Versão ONNX ({DTYPE}) para Transformers.js",
)
print(f"Enviado: https://huggingface.co/{HUB_REPO}/tree/main/onnx")
print(f'Na página da demo (index.html), use: const DTYPE = "{DTYPE}";')

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...onnx/model_quantized.onnx:   7%|7         | 8.00MB /  110MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


Enviado: https://huggingface.co/suellenlima/bertimbau-sentimento-reviews-ptbr/tree/main/onnx
Na página da demo (index.html), use: const DTYPE = "q8";
